# FM 5422 — Project 4: Treasury Future (TYH3)

This notebook computes, for each deliverable bond in **TYH3**:

- **Conversion Factor (CF)** (using QuantLib bond pricing at 6% yield)
- **Gross / Net Basis**
- **Implied Repo Rate**

Inputs are provided in the uploaded Excel files:
- `Bond Future DLV Data.xlsx`
- `Implied Repo Replication.xlsx`


In [166]:
# If QuantLib is not installed in your environment, uncomment:
# !pip -q install QuantLib-Python

import re
import math
import numpy as np
import pandas as pd
import QuantLib as ql


In [167]:
# -----------------------------
# Project inputs (from Excel)
# -----------------------------
DLV_XLSX = "Bond Future DLV Data.xlsx"
REPO_XLSX = "Implied Repo Replication.xlsx"

# TYH3 settings (from the replication sheet / Bloomberg screenshot)
SETTLEMENT_DATE = ql.Date(7, 2, 2023)   # 2023-02-07
DELIVERY_DATE   = ql.Date(31, 3, 2023)  # 2023-03-31 (last delivery date used in the replication)

# Futures price (per $100 notional). 113-16 = 113.5
FUTURES_PRICE = 113.5


In [168]:
# -----------------------------
# Helpers: parse coupon, maturity, and bond price quotes
# -----------------------------
def frac_to_float(s: str) -> float:
    """Convert strings like '3 ⅞' to 3.875"""
    s = str(s).strip()
    s = (s.replace('½', ' 1/2')
           .replace('¼', ' 1/4')
           .replace('¾', ' 3/4')
           .replace('⅛', ' 1/8')
           .replace('⅜', ' 3/8')
           .replace('⅝', ' 5/8')
           .replace('⅞', ' 7/8'))
    parts = s.split()
    total = 0.0
    for p in parts:
        if '/' in p:
            a, b = p.split('/')
            total += float(a) / float(b)
        else:
            total += float(p)
    return total


def parse_security(name: str):
    """Parse 'T 3 ⅞ 09/30/29' -> (coupon_rate_decimal, maturity_qlDate)"""
    name = str(name)
    m = re.search(r'T\s+(.+?)\s+(\d{2}/\d{2}/\d{2})', name)
    if not m:
        raise ValueError(f"Cannot parse security name: {name}")
    coupon_str = m.group(1).strip()
    mat_str = m.group(2)
    coupon_rate = frac_to_float(coupon_str) / 100.0

    mm, dd, yy = map(int, mat_str.split('/'))
    maturity = ql.Date(dd, mm, 2000 + yy)
    return coupon_rate, maturity


def parse_price(p):
    """Parse treasury price strings like '101-10+' or '100-20 3/4' into decimal clean price."""
    if isinstance(p, (int, float, np.number)):
        return float(p)

    s = str(p).strip().replace(',', '')

    # 101-10+  -> 101 + 10/32 + 1/64
    m = re.match(r'^(\d+)-(\d+)(\+)?$', s)
    if m:
        pts = int(m.group(1))
        thirtyseconds = int(m.group(2))
        plus = m.group(3)
        return pts + thirtyseconds/32.0 + (1/64.0 if plus else 0.0)

    # 100-20 3/4 -> 100 + (20 + 3/4)/32
    m = re.match(r'^(\d+)-(\d+)\s+(\d+)/(\d+)$', s)
    if m:
        pts = int(m.group(1))
        thirtyseconds = int(m.group(2))
        num = int(m.group(3))
        den = int(m.group(4))
        return pts + (thirtyseconds + num/den) / 32.0

    # 100-18 -> 100 + 18/32
    m = re.match(r'^(\d+)-(\d+)$', s)
    if m:
        pts = int(m.group(1))
        thirtyseconds = int(m.group(2))
        return pts + thirtyseconds/32.0

    # fallback float
    return float(s)


In [169]:
# -----------------------------
# Load inputs
# -----------------------------
df_dlv = pd.read_excel(DLV_XLSX, sheet_name=0).copy()
df_dlv


,Cash Security,IssueDate,Price,Conver Factor,Gro/Bas (32nds)
0,T 4 10/31/29,2022-10-31,101-10+,0.8937,-3.4176
1,T 3 ⅞ 09/30/29,2022-09-30,100-18,0.8870,-3.5840
2,T 3 ⅞ 11/30/29,2022-11-30,100-20 3/4,0.8870,-0.8336
3,T 3 ⅞ 12/31/29,2023-01-03,100-23 3/4,0.8834,NaN
4,T 3 ½ 01/31/30,2023-01-31,98-15 3/4,0.8628,NaN
5,T 1 ¾ 11/15/29,2019-11-15,88-19,0.7740,NaN
6,T 1 ½ 02/15/30,2020-02-18,86-23,0.7532,NaN
7,T 0 ⅝ 05/15/30,2020-05-15,80-23+,0.6964,NaN
8,T 0 ⅝ 08/15/30,2020-08-17,80-10+,0.6877,NaN
9,T 0 ⅞ 11/15/30,2020-11-16,81-18+,0.6941,NaN


In [170]:
# -----------------------------
# QuantLib conventions
# -----------------------------
calendar  = ql.UnitedStates(ql.UnitedStates.GovernmentBond)
day_count = ql.ActualActual(ql.ActualActual.Bond)
comp      = ql.Compounded
freq      = ql.Semiannual

# Set evaluation date (for consistency in any date-dependent calculations)
ql.Settings.instance().evaluationDate = SETTLEMENT_DATE


In [171]:
# -----------------------------
# Build a bond object for each deliverable
#
# NOTE:
# The Excel column 'IssueDate' is used as the bond's start (effective) date.
# That's sufficient for accurate accrued interest and cashflows between settlement and delivery.
# -----------------------------
def make_bond(effective_date: ql.Date, maturity: ql.Date, coupon_rate: float) -> ql.FixedRateBond:
    schedule = ql.Schedule(
        effective_date,
        maturity,
        ql.Period(freq),
        calendar,
        ql.Unadjusted,
        ql.Unadjusted,
        ql.DateGeneration.Backward,
        True
    )
    return ql.FixedRateBond(
        0,
        100.0,
        schedule,
        [coupon_rate],
        day_count,
        ql.Following
    )


def accrued_on(bond: ql.Bond, d: ql.Date) -> float:
    original = ql.Settings.instance().evaluationDate
    ql.Settings.instance().evaluationDate = d
    ai = bond.accruedAmount()
    ql.Settings.instance().evaluationDate = original
    return ai


def coupon_carry(bond: ql.Bond) -> float:
    return sum(
        cf.amount()
        for cf in bond.cashflows()
        if cf.date() > SETTLEMENT_DATE and cf.date() <= DELIVERY_DATE
    )


In [172]:
df = df_dlv.copy()

# Parse coupon & maturity
parsed = df["Cash Security"].apply(parse_security)
df["Coupon"] = parsed.apply(lambda x: x[0])
df["Maturity"] = parsed.apply(lambda x: x[1])

# Parse clean price
df["CleanPrice"] = df["Price"].apply(parse_price)

# Convert issue date
df["IssueDate_ql"] = df["IssueDate"].apply(
    lambda d: ql.Date(d.day, d.month, d.year)
)

In [173]:
df["Bond"] = df.apply(
    lambda r: make_bond(r["IssueDate_ql"], r["Maturity"], r["Coupon"]),
    axis=1
)

In [174]:
df["AI_Settle"] = df["Bond"].apply(lambda b: accrued_on(b, SETTLEMENT_DATE))

df["AI_Delivery"] = df["Bond"].apply(lambda b: accrued_on(b, DELIVERY_DATE))

df["Dirty_Settle"] = df["CleanPrice"] + df["AI_Settle"]


In [175]:
df["Futures_x_CF"] = FUTURES_PRICE * df["Conver Factor"]

df["Invoice"] = df["Futures_x_CF"] + df["AI_Delivery"]

In [176]:
df["CouponCarry"] = df["Bond"].apply(coupon_carry)

In [177]:
df["GrossBasis"] = df["CleanPrice"] - df["Futures_x_CF"]

In [178]:
# Implied repo calculation
days = DELIVERY_DATE - SETTLEMENT_DATE

df["ImpliedRepo"] = ((df["Invoice"] + df["CouponCarry"] - df["Dirty_Settle"])/df["Dirty_Settle"]*(360.0 / days))*100

In [179]:
## Given market repo rate

market_repo = 0.04597

repo_cost = df["Dirty_Settle"] * market_repo * (days / 360.0)

df["NetBasis"] = (df["Dirty_Settle"] + repo_cost - df["CouponCarry"]) - df["Invoice"]

In [180]:
df["GrossBasis_32nds"] = df["GrossBasis"] * 32
df["NetBasis_32nds"] = df["NetBasis"] * 32

In [181]:
df[["Cash Security", "CleanPrice", "Conver Factor", "Futures_x_CF", "Invoice", "CouponCarry", "Dirty_Settle", "GrossBasis_32nds", "NetBasis_32nds", "ImpliedRepo"]]

,Cash Security,CleanPrice,Conver Factor,Futures_x_CF,Invoice,CouponCarry,Dirty_Settle,GrossBasis_32nds,NetBasis_32nds,ImpliedRepo
0,T 4 10/31/29,101.328125,0.8937,101.43495,103.103458,0.0000,102.422048,-3.4184,-0.042139,4.605901
1,T 3 ⅞ 09/30/29,100.562500,0.8870,100.67450,100.674500,1.9375,101.946429,-3.5840,0.363654,4.519827
2,T 3 ⅞ 11/30/29,100.648438,0.8870,100.67450,101.962618,0.0000,101.382984,-0.8340,2.993931,3.958110
3,T 3 ⅞ 12/31/29,100.742188,0.8834,100.26590,101.197185,0.0000,101.116842,15.2412,18.914711,0.550073
4,T 3 ½ 01/31/30,98.492188,0.8628,97.92780,98.498242,0.0000,98.559867,18.0604,22.914353,-0.432869
5,T 1 ¾ 11/15/29,88.593750,0.7740,87.84900,88.506459,0.0000,88.999827,23.8320,34.698801,-3.837794
6,T 1 ½ 02/15/30,86.718750,0.7532,85.48820,85.670520,0.7500,87.436141,39.3776,51.078610,-8.041550
7,T 0 ⅝ 05/15/30,80.734375,0.6964,79.04140,79.276207,0.0000,80.879403,54.1752,68.487815,-13.722961
8,T 0 ⅝ 08/15/30,80.328125,0.6877,78.05395,78.129917,0.3125,80.627038,72.7736,87.039798,-18.758348
9,T 0 ⅞ 11/15/30,81.578125,0.6941,78.78035,79.109079,0.0000,81.781164,89.5288,102.883853,-22.620179
